# 🐱🐶 CAIS Workshop: Build a CNN Image Classifier

**Carleton AI Society | Presented by Marc Aoun**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/carletonaiadmin/model/blob/main/CAIS_Workshop_CNN_Classifier.ipynb)

Can a neural network learn to tell the difference between a **cat** and a **dog** just by looking at a picture?

Today we'll build a **Convolutional Neural Network (CNN)** from scratch, the same architecture used in self-driving cars, medical imaging, and facial recognition.

---
**How to use this notebook:**
- ⚡ First: Go to **Runtime → Change runtime type → GPU** (makes training way faster!)
- Run each cell one at a time with **Shift + Enter**
- Read the output before moving to the next cell
---

## 📦 Step 1: Import Libraries

| Library | What it does |
|---------|-------------|
| `tensorflow / keras` | Build and train neural networks |
| `numpy` | Math operations on arrays |
| `matplotlib` | Create charts and display images |
| `sklearn` | Evaluation metrics (accuracy, confusion matrix) |


In [ ]:
# --- STEP 1: Import Libraries ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print(f"✅ TensorFlow version: {tf.__version__}")
print(f"🖥️  GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
if len(tf.config.list_physical_devices('GPU')) > 0:
    print(f"   GPU: {tf.config.list_physical_devices('GPU')[0]}")
else:
    print("   ⚠️  No GPU detected — go to Runtime → Change runtime type → GPU")


## 📊 Step 2: Load the Data

We're using **CIFAR-10** — a famous dataset of 60,000 tiny images across 10 categories.

We'll filter it down to just **cats (class 3)** and **dogs (class 5)** for binary classification.

This gives us a real image classification task that matches the CNN architecture on the slides.


In [ ]:
# --- STEP 2: Load the Data ---
# CIFAR-10: 60,000 32x32 color images in 10 classes
(X_train_full, y_train_full), (X_test_full, y_test_full) = keras.datasets.cifar10.load_data()

# Class names for reference
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Filter to CATS (3) and DOGS (5) only
cat_dog_train = np.isin(y_train_full.flatten(), [3, 5])
cat_dog_test = np.isin(y_test_full.flatten(), [3, 5])

X_train = X_train_full[cat_dog_train]
y_train = y_train_full[cat_dog_train].flatten()
X_test = X_test_full[cat_dog_test]
y_test = y_test_full[cat_dog_test].flatten()

# Convert labels: Cat = 0, Dog = 1
y_train = (y_train == 5).astype(int)  # Dog = 1
y_test = (y_test == 5).astype(int)

label_names = ["Cat 🐱", "Dog 🐶"]

print(f"Training images: {len(X_train)}")
print(f"Testing images:  {len(X_test)}")
print(f"Image shape: {X_train[0].shape} (32x32 pixels, 3 color channels)")
print(f"\n  🐱 Cats in training: {(y_train == 0).sum()}")
print(f"  🐶 Dogs in training: {(y_train == 1).sum()}")


## 🔍 Step 3: Explore the Data

Before training, let's actually **look** at what the model will see.

These are 32×32 pixel images — tiny! The fact that a CNN can classify these accurately is impressive.


In [ ]:
# --- STEP 3: Explore the Data ---
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))

# Show 8 cats
cat_indices = np.where(y_train == 0)[0][:8]
for i, idx in enumerate(cat_indices):
    axes[0, i].imshow(X_train[idx])
    axes[0, i].set_title("Cat 🐱", fontsize=10)
    axes[0, i].axis("off")

# Show 8 dogs
dog_indices = np.where(y_train == 1)[0][:8]
for i, idx in enumerate(dog_indices):
    axes[1, i].imshow(X_train[idx])
    axes[1, i].set_title("Dog 🐶", fontsize=10)
    axes[1, i].axis("off")

plt.suptitle("Sample Training Images (32×32 pixels)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nThese images are TINY — 32×32 pixels. That's smaller than most app icons.")
print("The CNN needs to learn to distinguish cats from dogs with just 1,024 pixels per channel!")


## ⚙️ Step 4: Preprocess the Data

Two things we need to do:

1. **Normalize pixel values** from 0-255 → 0-1 (puts everything on equal footing)
2. **Check the label balance** (are cats and dogs roughly equal?)

This is the same preprocessing step from the pipeline — clean the data before feeding it to the model.


In [ ]:
# --- STEP 4: Preprocess ---

# Normalize pixel values: 0-255 → 0.0-1.0
X_train_scaled = X_train.astype("float32") / 255.0
X_test_scaled = X_test.astype("float32") / 255.0

print("BEFORE normalization:")
print(f"  Pixel range: {X_train[0].min()} to {X_train[0].max()}")
print(f"\nAFTER normalization:")
print(f"  Pixel range: {X_train_scaled[0].min():.1f} to {X_train_scaled[0].max():.1f}")

print(f"\n✅ {len(X_train_scaled)} training images normalized")
print(f"✅ {len(X_test_scaled)} test images normalized")
print(f"\nLabel balance:")
print(f"  🐱 Cats: {(y_train == 0).sum()} ({(y_train == 0).mean()*100:.1f}%)")
print(f"  🐶 Dogs: {(y_train == 1).sum()} ({(y_train == 1).mean()*100:.1f}%)")
print(f"  → Perfectly balanced! ✅")


## 🏗️ Step 5: Build the CNN Architecture

This is the exact architecture from **Slide 7**:

| Layer | What it does |
|-------|-------------|
| **Conv2D** | Extracts features (edges, textures, shapes) |
| **ReLU** | Activation — removes negative values, adds non-linearity |
| **MaxPooling** | Shrinks the image, keeps important features |
| **Flatten** | Converts 2D feature maps → 1D vector |
| **Dense** | Fully connected layer — combines features for decision |
| **Output** | Final prediction — Cat or Dog probability |

We stack these layers to go from **raw pixels → prediction**.


In [ ]:
# --- STEP 5: Build the CNN ---

model = keras.Sequential([
    # --- LAYER 1: First Convolution Block ---
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    # 32 filters, each 3×3 pixels. Detects basic features like edges.

    layers.MaxPooling2D((2, 2)),
    # Shrinks from 30×30 → 15×15. Keeps strongest features.

    # --- LAYER 2: Second Convolution Block ---
    layers.Conv2D(64, (3, 3), activation='relu'),
    # 64 filters now — detects more complex patterns (shapes, textures).

    layers.MaxPooling2D((2, 2)),
    # Shrinks from 13×13 → 6×6.

    # --- LAYER 3: Third Convolution Block ---
    layers.Conv2D(64, (3, 3), activation='relu'),
    # Even more complex features (ears, fur patterns, snouts).

    # --- FLATTEN: 2D → 1D ---
    layers.Flatten(),
    # Converts the 2D feature maps into a single long vector.

    # --- FULLY CONNECTED ---
    layers.Dense(64, activation='relu'),
    # Combines all features to form a decision.

    # --- OUTPUT ---
    layers.Dense(1, activation='sigmoid'),
    # Single neuron: outputs probability between 0 (cat) and 1 (dog).
])

# Configure training
model.compile(
    optimizer='adam',                        # Adam optimizer — adapts learning rate
    loss='binary_crossentropy',             # Standard for binary classification
    metrics=['accuracy']                     # Track accuracy during training
)

# Show the architecture
model.summary()

# Count parameters
total_params = model.count_params()
print(f"\n🧠 Total trainable parameters: {total_params:,}")
print("   Each parameter is a number the model will learn during training!")


## 🏋️ Step 6: Train the Model

**This is where the learning happens.**

The model will:
1. Look at a batch of images
2. Make predictions (mostly wrong at first)
3. Calculate how wrong it was (loss)
4. Adjust its weights to be less wrong
5. Repeat for every batch, for multiple **epochs**

Watch the accuracy climb with each epoch!

⏱️ *This takes about 1-2 minutes with GPU enabled.*


In [ ]:
# --- STEP 6: Train the Model ---

print("🏋️ Training started... watch the accuracy climb!\n")

history = model.fit(
    X_train_scaled, y_train,
    epochs=10,                              # 10 passes through the full dataset
    batch_size=64,                          # Process 64 images at a time
    validation_split=0.2,                   # Hold out 20% for validation
    verbose=1                               # Show progress
)

print("\n✅ Training complete!")


## 📈 Step 7: Visualize Training Progress

These curves show how the model improved over time.

- **Training accuracy** should go up (the model is learning)
- **Validation accuracy** should also go up (the model is generalizing)
- If training goes up but validation goes DOWN → **overfitting!** (memorizing, not learning)


In [ ]:
# --- STEP 7: Training Curves ---

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history.history['accuracy'], label='Training', linewidth=2, color='#2ecc71')
ax1.plot(history.history['val_accuracy'], label='Validation', linewidth=2, color='#e74c3c')
ax1.set_title('Accuracy Over Time', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend(fontsize=12)
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(history.history['loss'], label='Training', linewidth=2, color='#2ecc71')
ax2.plot(history.history['val_loss'], label='Validation', linewidth=2, color='#e74c3c')
ax2.set_title('Loss Over Time', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
print(f"Final training accuracy:   {final_train_acc*100:.1f}%")
print(f"Final validation accuracy: {final_val_acc*100:.1f}%")

if final_train_acc - final_val_acc > 0.10:
    print("\n⚠️  Gap between training and validation → signs of overfitting!")
else:
    print("\n✅ Training and validation are close — model is generalizing well!")


## 📊 Step 8: Evaluate on Test Data

**Moment of truth.** Let's test the model on images it has **NEVER seen** during training.

This is the honest, unbiased evaluation.


In [ ]:
# --- STEP 8: Evaluate ---

test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\n✅ Test Accuracy: {test_accuracy * 100:.1f}%")
print(f"   That means {int(test_accuracy * len(y_test))}/{len(y_test)} images correctly classified!\n")

# Predictions for confusion matrix
y_pred_proba = model.predict(X_test_scaled, verbose=0).flatten()
y_pred = (y_pred_proba >= 0.5).astype(int)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Cat 🐱", "Dog 🐶"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix — Cat vs Dog CNN", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nHow to read this:")
print(f"  ✅ Top-left ({cm[0][0]}): Correctly identified CATS")
print(f"  ✅ Bottom-right ({cm[1][1]}): Correctly identified DOGS")
print(f"  ❌ Top-right ({cm[0][1]}): Cat misclassified as Dog")
print(f"  ❌ Bottom-left ({cm[1][0]}): Dog misclassified as Cat")


## 🔮 Step 9: See the Model's Predictions

Let's look at actual test images and see what the model predicts — with confidence scores.

🟢 = Correct prediction | 🔴 = Wrong prediction


In [ ]:
# --- STEP 9: Show Predictions ---

fig, axes = plt.subplots(2, 6, figsize=(18, 6))

# Pick 12 random test images
np.random.seed(42)
indices = np.random.choice(len(X_test), 12, replace=False)

for i, idx in enumerate(indices):
    row, col = i // 6, i % 6
    ax = axes[row, col]

    ax.imshow(X_test[idx])

    true_label = label_names[y_test[idx]]
    pred_label = label_names[y_pred[idx]]
    confidence = y_pred_proba[idx] if y_pred[idx] == 1 else 1 - y_pred_proba[idx]

    correct = y_test[idx] == y_pred[idx]
    color = "#2ecc71" if correct else "#e74c3c"
    emoji = "✅" if correct else "❌"

    ax.set_title(f"{emoji} {pred_label}\n{confidence*100:.0f}% confident",
                 fontsize=10, color=color, fontweight="bold")
    ax.axis("off")

plt.suptitle("Model Predictions on Test Images", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

correct_count = (y_pred == y_test).sum()
print(f"\nIn this sample: {correct_count}/12 correct")


## 🎯 Step 10: YOU Pick an Image!

Pick a number between **0 and 1999** and let's see if the CNN can tell if it's a cat or dog!

Change the number below and re-run to try different images.


In [ ]:
# --- STEP 10: Pick an Image! ---
# 👇 CHANGE THIS NUMBER AND RE-RUN! 👇
IMAGE_NUMBER = 1000

# Get the image
img = X_test[IMAGE_NUMBER]
img_scaled = X_test_scaled[IMAGE_NUMBER:IMAGE_NUMBER+1]

# Predict
prediction = model.predict(img_scaled, verbose=0).flatten()[0]
predicted_class = "Dog 🐶" if prediction >= 0.5 else "Cat 🐱"
true_class = label_names[y_test[IMAGE_NUMBER]]
confidence = prediction if prediction >= 0.5 else 1 - prediction

# Display
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(img)
ax.axis("off")
correct = predicted_class.split()[0] == true_class.split()[0]
ax.set_title(f"Image #{IMAGE_NUMBER}", fontsize=14, fontweight="bold")
plt.show()

print(f"\n🤖 CNN Prediction:  {predicted_class}")
print(f"📊 Confidence:     {confidence * 100:.1f}%")
print(f"🏷️  Actual label:   {true_class}")
print(f"{'✅ CORRECT!' if correct else '❌ WRONG!'}")
print(f"\n→ Change IMAGE_NUMBER above and re-run to try another!")


## 🔄 BONUS: What Happens With More/Fewer Epochs?

Overfitting is when the model **memorizes** training data instead of **learning** patterns.

Try changing the epochs below to see the effect:
- `epochs=2` → Underfitting (hasn't learned enough)
- `epochs=10` → Good balance
- `epochs=50` → Likely overfitting (watch the gap between training and validation)


In [ ]:
# --- BONUS: Experiment with Epochs ---
# 👇 CHANGE THIS AND RE-RUN 👇
EPOCHS = 30  # Try: 2, 5, 10, 30, 50

# Build a fresh model (same architecture)
bonus_model = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])

bonus_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print(f"Training for {EPOCHS} epochs...\n")
bonus_history = bonus_model.fit(X_train_scaled, y_train, epochs=EPOCHS,
                                 batch_size=64, validation_split=0.2, verbose=1)

bonus_acc = bonus_model.evaluate(X_test_scaled, y_test, verbose=0)[1]
print(f"\n📊 Test accuracy with {EPOCHS} epochs: {bonus_acc*100:.1f}%")
print(f"   Compare to our original 10-epoch model: {test_accuracy*100:.1f}%")

if EPOCHS <= 3:
    print("\n💡 Too few epochs → model hasn't learned enough (underfitting)")
elif EPOCHS >= 30:
    print("\n💡 Many epochs → check if validation accuracy stopped improving (overfitting risk)")
